# Tái lập bài báo: *New Stocks Prediction Portfolio Optimization on VN30: Machine Learning Method*

Nguyễn Anh Phong & Nguyễn Trúc Linh, *Kurdish Studies* 11(2), 2023, tr. 5854–5862. DOI: 10.58262/ks.v11i2.425

Notebook này làm lại từng bước của bài, đánh số hình theo bài để tiện đối chiếu:

| Bước trong bài | Phần trong notebook |
|---|---|
| Lấy giá ngày VN30 qua `vnstock` (mục 3.1) | Phần 1–2 |
| Bước 1: gộp 30 mã thành một bảng giá đóng cửa | Phần 3 |
| Bước 2: lợi suất trung bình theo ngày; Bước 3: chọn top 5 | Phần 4 |
| Biến động ngày/năm (Hình 1–2), mật độ lợi suất, lợi suất theo ngày (Hình 3) | Phần 5 |
| Tăng trưởng nếu bỏ 2.000.000 VND vào từng mã (Hình 4) | Phần 6 |
| Tương quan 5 mã (Hình 5) | Phần 7 |
| 1000 danh mục ngẫu nhiên + đường biên hiệu quả + Sharpe tối đa (Hình 6) | Phần 8 |
| Trọng số (Hình 7), chỉ số kỳ vọng (Hình 8) | Phần 9 |
| Tăng trưởng 2.000.000 VND theo danh mục (Hình 9) | Phần 10 |
| **Mở rộng (bài không có):** kiểm tra ngoài mẫu, so với danh mục chia đều | Phần 11 |

**Những chỗ bài không nói rõ, notebook phải tự chọn (có thể đổi ở phần Cấu hình):**
- Khoảng thời gian: bài chỉ nói "từ khi niêm yết đến nay", bài nhận tháng 6/2023 → mặc định lấy đến 31/05/2023.
- Danh sách VN30: dùng rổ VN30 đầu năm 2023 (có BCM, khớp với top 5 trong bài).
- Lãi suất phi rủi ro: bài không ghi → mặc định 2%/năm (mặc định của thư viện PyPortfolioOpt mà bài nhiều khả năng đã dùng).
- Lợi suất kỳ vọng năm = lợi suất ngày trung bình × 252; độ biến động năm = độ lệch chuẩn ngày × √252 (đúng như bài mô tả).

## 1. Cấu hình

In [ ]:
# Cài thư viện (chạy được cả trên Google Colab lẫn Jupyter trên máy)
%pip install -q yfinance pyportfolioopt requests

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# Rổ VN30 đầu năm 2023 (sửa nếu muốn rổ khác)
VN30 = ["ACB", "BCM", "BID", "BVH", "CTG", "FPT", "GAS", "GVR", "HDB", "HPG",
        "MBB", "MSN", "MWG", "NVL", "PDR", "PLX", "POW", "SAB", "SSI", "STB",
        "TCB", "TPB", "VCB", "VHM", "VIB", "VIC", "VJC", "VNM", "VPB", "VRE"]

START = "2010-01-01"      # bài: "từ khi bắt đầu giao dịch" -> lấy càng xa càng tốt
END = "2023-05-31"        # bài nhận tháng 6/2023
TOP_N = 5                 # bài chọn 5 mã
N_PORTFOLIOS = 1000       # bài tạo 1000 danh mục ngẫu nhiên
RISK_FREE = 0.02          # bài không ghi; giả định 2%/năm
TRADING_DAYS = 252
INITIAL_VND = 2_000_000   # số tiền bài dùng ở Hình 4 và Hình 9
SEED = 42

# Nguồn dữ liệu: "auto" thử lần lượt dnse -> cafef -> vnstock -> yfinance -> csv
# "dnse": API biểu đồ của DNSE (giá đã điều chỉnh, lịch sử dài, không cần cài thêm) - nên dùng
# "cafef": dữ liệu lịch sử CafeF (cột GiaDieuChinh = giá điều chỉnh)
# "yfinance": Yahoo, lịch sử ngắn (VIB, ACB, BCM chỉ từ 2020-2021) và có lỗi giá, chỉ dùng khi không có nguồn khác
# "csv": đọc thư mục data/prices/<MÃ>.csv (cột date/time và close)
# "demo": dữ liệu MÔ PHỎNG, chỉ để kiểm tra code chạy được, KHÔNG dùng để kết luận
DATA_SOURCE = os.environ.get("VN30_DATA_SOURCE", "auto")
CSV_DIR = "data/prices"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)

## 2. Tải dữ liệu

Bài dùng `vnstock`, nhưng thư viện này hiện không cài được từ PyPI. Notebook tự lấy giá trực tiếp, thử lần lượt:
1. **DNSE** (`services.entrade.com.vn`): giá đã điều chỉnh cổ tức/chia tách, có cả giai đoạn mã còn ở UPCoM/HNX. Đây là nguồn chính.
2. **CafeF**: cột `GiaDieuChinh`.
3. `vnstock` (nếu bạn tự cài được), 4. `yfinance` (Yahoo, lịch sử ngắn và có lỗi giá), 5. file CSV trong `data/prices/`.

Đơn vị giá (nghìn VND hay VND) không ảnh hưởng lợi suất. Mỗi nguồn có file lưu tạm riêng `data/vn30_close_<nguồn>.csv`, nên bảng giá Yahoo cũ không bị dùng nhầm. Muốn tải lại từ đầu thì xóa file đó.

In [ ]:
def _from_vnstock(ticker, start, end):
    try:  # vnstock >= 3.x
        from vnstock import Vnstock
        df = Vnstock().stock(symbol=ticker, source="VCI").quote.history(start=start, end=end, interval="1D")
        return df.set_index(pd.to_datetime(df["time"]))["close"]
    except ImportError:
        raise
    except Exception:
        from vnstock import stock_historical_data  # vnstock 0.x (2023)
        df = stock_historical_data(symbol=ticker, start_date=start, end_date=end, resolution="1D", type="stock")
        col = "time" if "time" in df.columns else "TradingDate"
        return df.set_index(pd.to_datetime(df[col]))["close"]


def load_vnstock(tickers, start, end):
    import vnstock  # noqa: F401  (báo lỗi ngay nếu chưa cài)
    out = {}
    for t in tickers:
        try:
            out[t] = _from_vnstock(t, start, end)
        except Exception as e:
            print(f"  vnstock lỗi {t}: {e}")
    return pd.DataFrame(out)


HEADERS = {"User-Agent": "Mozilla/5.0"}


def load_dnse(tickers, start, end):
    import requests
    t0 = int(pd.Timestamp(start, tz="Asia/Ho_Chi_Minh").timestamp())
    t1 = int((pd.Timestamp(end, tz="Asia/Ho_Chi_Minh") + pd.Timedelta(days=1)).timestamp())
    out = {}
    for t in tickers:
        try:
            r = requests.get("https://services.entrade.com.vn/chart-api/v2/ohlcs/stock",
                             params={"from": t0, "to": t1, "symbol": t, "resolution": "1D"},
                             headers=HEADERS, timeout=30)
            j = r.json()
            if not j.get("t"):
                print(f"  dnse: không có dữ liệu {t}")
                continue
            idx = (pd.to_datetime(j["t"], unit="s", utc=True)
                   .tz_convert("Asia/Ho_Chi_Minh").normalize().tz_localize(None))
            out[t] = pd.Series(j["c"], index=idx, dtype=float).groupby(level=0).last()
        except Exception as e:
            print(f"  dnse lỗi {t}: {e}")
    return pd.DataFrame(out)


def load_cafef(tickers, start, end, page_size=1000):
    import requests
    url = "https://cafef.vn/du-lieu/Ajax/PageNew/DataHistory/PriceHistory.ashx"
    s_, e_ = pd.Timestamp(start), pd.Timestamp(end)
    out = {}
    for t in tickers:
        rows, page = [], 1
        while page <= 50:
            j = requests.get(url, params={"Symbol": t, "StartDate": s_.strftime("%d/%m/%Y"),
                                          "EndDate": e_.strftime("%d/%m/%Y"), "PageIndex": page,
                                          "PageSize": page_size}, headers=HEADERS, timeout=30).json()
            batch = (j.get("Data") or {}).get("Data") or []
            if not batch:
                break
            rows += batch
            if pd.to_datetime(batch[-1]["Ngay"], dayfirst=True) <= s_ or len(batch) < page_size:
                break
            page += 1
        if rows:
            df = pd.DataFrame(rows)
            out[t] = (df.assign(Ngay=pd.to_datetime(df["Ngay"], dayfirst=True))
                        .drop_duplicates("Ngay").set_index("Ngay")["GiaDieuChinh"].astype(float).sort_index())
        else:
            print(f"  cafef: không có dữ liệu {t}")
    return pd.DataFrame(out)


def load_yfinance(tickers, start, end):
    import yfinance as yf
    raw = yf.download([t + ".VN" for t in tickers], start=start, end=end,
                      auto_adjust=True, progress=False)["Close"]
    raw.columns = [c.replace(".VN", "") for c in raw.columns]
    return raw


def load_csv(tickers, folder):
    out = {}
    for t in tickers:
        path = os.path.join(folder, f"{t}.csv")
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        df.columns = [c.lower() for c in df.columns]
        dcol = next(c for c in ["date", "time", "tradingdate"] if c in df.columns)
        out[t] = df.set_index(pd.to_datetime(df[dcol]))["close"]
    return pd.DataFrame(out)


def load_demo(tickers, start, end, seed=SEED):
    # Giá MÔ PHỎNG (chuyển động Brown hình học có tương quan). Chỉ để chạy thử code.
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end)
    n, k = len(dates), len(tickers)
    mu = rng.uniform(0.0000, 0.0012, k)
    sig = rng.uniform(0.015, 0.03, k)
    corr = 0.45 + 0.1 * rng.random((k, k)); corr = (corr + corr.T) / 2; np.fill_diagonal(corr, 1)
    L = np.linalg.cholesky(corr)
    r = mu + (rng.standard_normal((n, k)) @ L.T) * sig
    px = pd.DataFrame(10_000 * np.exp(np.cumsum(r, axis=0)), index=dates, columns=tickers)
    for t in tickers:  # giả lập ngày niêm yết khác nhau
        px.loc[: dates[rng.integers(0, n // 2)], t] = np.nan
    return px


loaders = {"dnse": lambda: load_dnse(VN30, START, END),
           "cafef": lambda: load_cafef(VN30, START, END),
           "vnstock": lambda: load_vnstock(VN30, START, END),
           "yfinance": lambda: load_yfinance(VN30, START, END),
           "csv": lambda: load_csv(VN30, CSV_DIR),
           "demo": lambda: load_demo(VN30, START, END)}
order = ["dnse", "cafef", "vnstock", "yfinance", "csv"] if DATA_SOURCE == "auto" else [DATA_SOURCE]

prices, used = pd.DataFrame(), None
for src in order:
    cache = f"data/vn30_close_{src}.csv"
    try:
        if os.path.exists(cache) and src != "demo":
            prices = pd.read_csv(cache, index_col=0, parse_dates=True)
        else:
            prices = loaders[src]().dropna(axis=1, how="all")
            if src != "demo" and prices.shape[1] >= TOP_N:
                os.makedirs("data", exist_ok=True)
                prices.to_csv(cache)
        if prices.shape[1] >= TOP_N:
            used = src
            break
        print(f"{src}: chỉ lấy được {prices.shape[1]} mã, thử nguồn tiếp theo")
    except Exception as e:
        print(f"{src}: không dùng được ({type(e).__name__}: {e})")

if used is None:
    raise RuntimeError("Không tải được dữ liệu từ nguồn nào. Đặt file CSV vào data/prices/ rồi chạy lại.")

prices = prices.sort_index().loc[START:END]
print(f"Nguồn: {used} | {prices.shape[1]} mã | {prices.index.min().date()} -> {prices.index.max().date()}")
if used == "demo":
    print("CẢNH BÁO: đang dùng dữ liệu MÔ PHỎNG, kết quả bên dưới không phải thị trường thật.")

## 3. Bước 1: gộp 30 mã thành một bảng giá đóng cửa

Mỗi cột là một mã, mỗi dòng là một ngày giao dịch. Các mã niêm yết muộn có giá trị trống ở đầu bảng.

In [ ]:
missing = sorted(set(VN30) - set(prices.columns))
if missing:
    print("Thiếu mã:", missing)
print("Ngày bắt đầu có dữ liệu của từng mã (10 mã niêm yết muộn nhất):")
display(prices.apply(lambda s: s.first_valid_index()).sort_values().tail(10).dt.date.to_frame("ngày đầu"))
prices.tail()

### Kiểm tra chất lượng dữ liệu

Sàn HOSE giới hạn biên độ ±7%/phiên (lúc mới niêm yết hoặc ở UPCoM/HNX biên độ rộng hơn), nên lợi suất ngày quá ±15% thường là lỗi dữ liệu hoặc giá chưa điều chỉnh. Giá đứng yên nhiều phiên liền cũng là dấu hiệu lỗi (lần chạy với Yahoo gặp cả hai).

In [ ]:
r_chk = prices.pct_change(fill_method=None)
big_moves = (r_chk.abs() > 0.15).sum()
stale = prices.apply(lambda s: (s.diff() == 0).astype(int).groupby((s.diff() != 0).cumsum()).sum().max())
qc = pd.DataFrame({"số phiên biến động >15%": big_moves, "chuỗi giá đứng yên dài nhất": stale})
display(qc[(qc.iloc[:, 0] > 0) | (qc.iloc[:, 1] >= 5)].sort_values("số phiên biến động >15%", ascending=False))
print("Bảng trống nghĩa là không phát hiện bất thường.")

## 4. Bước 2–3: lợi suất trung bình theo ngày, chọn top 5

Lợi suất ngày $r_t = P_t / P_{t-1} - 1$. Bài xếp hạng theo lợi suất ngày trung bình và lấy 5 mã cao nhất. Mỗi mã tính trên toàn bộ lịch sử của chính nó, đúng như cách bài mô tả.

> Bài ra top 5 là FPT, VIB, ACB, BCM, BID. Nếu dữ liệu thật của bạn ra khác, nguyên nhân thường là khoảng thời gian hoặc giá điều chỉnh/chưa điều chỉnh cổ tức.

In [ ]:
returns_all = prices.pct_change(fill_method=None)
mean_daily = returns_all.mean().sort_values(ascending=False)
top = mean_daily.head(TOP_N).index.tolist()
print("Top", TOP_N, ":", top)
mean_daily.to_frame("lợi suất ngày TB").assign(**{"lợi suất năm (x252)": lambda d: d.iloc[:, 0] * TRADING_DAYS}).head(10).style.format("{:.4%}")

## 5. Biến động (Hình 1–3)

- Độ biến động ngày = độ lệch chuẩn của lợi suất ngày.
- Độ biến động năm = độ biến động ngày × √252.

Từ đây trở đi chỉ dùng **những ngày cả 5 mã đều có giá**, để các phép tính danh mục so trên cùng một khoảng thời gian.

In [ ]:
px5 = prices[top].dropna()
ret5 = px5.pct_change().dropna()
print(f"Giai đoạn chung của top {TOP_N}: {px5.index.min().date()} -> {px5.index.max().date()} ({len(ret5)} phiên)")

vol = pd.DataFrame({"biến động ngày": ret5.std(), "biến động năm": ret5.std() * np.sqrt(TRADING_DAYS)}).sort_values("biến động ngày")
display(vol.style.format("{:.2%}"))

fig, ax = plt.subplots(figsize=(7, 3.5))
vol["biến động ngày"].plot.bar(ax=ax, color="#4C72B0")
ax.set_title("Hình 1–2. Độ biến động ngày của top 5"); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_daily_volatility.png"); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for t in top:
    sns.kdeplot(ret5[t], ax=ax, label=t)
ax.set_xlim(-0.08, 0.08); ax.set_title("Mật độ lợi suất ngày của top 5"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_density.png"); plt.show()

fig, axes = plt.subplots(len(top), 1, figsize=(10, 1.6 * len(top)), sharex=True)
for ax, t in zip(axes, top):
    ax.plot(ret5.index, ret5[t], lw=0.5); ax.set_ylabel(t)
axes[0].set_title("Hình 3. Lợi suất theo ngày của top 5")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_daily_returns.png"); plt.show()

## 6. Hình 4: nếu bỏ 2.000.000 VND vào từng mã

Giá trị khoản đầu tư = 2.000.000 × (1 + r₁)(1 + r₂)…, tức nhân dồn lợi suất theo ngày.

In [ ]:
growth = INITIAL_VND * (1 + ret5).cumprod()
ax = growth.plot(figsize=(10, 4.5), title=f"Hình 4. Giá trị {INITIAL_VND:,.0f} VND đầu tư vào từng mã")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e6:.0f}tr"))
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_growth_each_stock.png"); plt.show()
growth.iloc[-1].sort_values().to_frame("giá trị cuối kỳ (VND)").style.format("{:,.0f}")

## 7. Hình 5: tương quan giữa 5 mã

In [ ]:
corr = ret5.corr()
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Hình 5. Tương quan lợi suất ngày"); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig5_correlation.png"); plt.show()
print(f"Tương quan trung bình (ngoài đường chéo): {corr.values[np.triu_indices(len(top), 1)].mean():.2f}  (bài ghi khoảng 0,65)")

## 8. Hình 6: 1000 danh mục ngẫu nhiên, đường biên hiệu quả, danh mục Sharpe tối đa

Với vector trọng số $w$ (không bán khống, tổng bằng 1):
- Lợi suất kỳ vọng năm: $\mu_p = w^\top \mu$, với $\mu$ = lợi suất ngày TB × 252
- Độ biến động năm: $\sigma_p = \sqrt{w^\top \Sigma w}$, với $\Sigma$ = hiệp phương sai ngày × 252
- Sharpe: $(\mu_p - r_f) / \sigma_p$

Bài dùng thư viện Efficient Frontier của Python (nhiều khả năng là PyPortfolioOpt). Ở đây mình tự giải bằng `scipy.optimize` để thấy rõ bài toán. Ô cuối phần này có đoạn kiểm tra chéo bằng PyPortfolioOpt nếu bạn đã cài.

In [ ]:
mu = ret5.mean() * TRADING_DAYS
cov = ret5.cov() * TRADING_DAYS
n = len(top)

def perf(w, mu=mu, cov=cov, rf=RISK_FREE):
    r = float(w @ mu); v = float(np.sqrt(w @ cov @ w))
    return r, v, (r - rf) / v

bounds = [(0, 1)] * n
sum1 = {"type": "eq", "fun": lambda w: w.sum() - 1}

def max_sharpe(mu, cov, rf=RISK_FREE, bounds=None):
    k = len(mu); bounds = bounds or [(0, 1)] * k
    res = minimize(lambda w: -(w @ mu - rf) / np.sqrt(w @ cov @ w), np.ones(k) / k,
                   bounds=bounds, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], method="SLSQP")
    return res.x

def min_vol_for(target, mu, cov):
    k = len(mu)
    res = minimize(lambda w: w @ cov @ w, np.ones(k) / k, bounds=[(0, 1)] * k, method="SLSQP",
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1},
                                {"type": "eq", "fun": lambda w: w @ mu - target}])
    return res.x if res.success else None

# 1000 danh mục ngẫu nhiên
rng = np.random.default_rng(SEED)
W = rng.random((N_PORTFOLIOS, n)); W /= W.sum(axis=1, keepdims=True)
rand = pd.DataFrame([perf(w) for w in W], columns=["return", "volatility", "sharpe"])

# Đường biên hiệu quả
w_minvol = minimize(lambda w: w @ cov.values @ w, np.ones(n) / n, bounds=bounds, constraints=[sum1], method="SLSQP").x
targets = np.linspace(perf(w_minvol)[0], mu.max(), 60)
frontier = [perf(w) for w in (min_vol_for(t, mu.values, cov.values) for t in targets) if w is not None]
frontier = pd.DataFrame(frontier, columns=["return", "volatility", "sharpe"])

w_ms = max_sharpe(mu.values, cov.values)
r_ms, v_ms, s_ms = perf(w_ms)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(rand["volatility"], rand["return"], c=rand["sharpe"], cmap="viridis", s=10, alpha=0.7)
plt.colorbar(sc, label="Sharpe")
ax.plot(frontier["volatility"], frontier["return"], "k-", lw=2, label="Đường biên hiệu quả")
ax.scatter(v_ms, r_ms, marker="*", s=350, c="red", label="Sharpe tối đa")
for t in top:
    ax.scatter(np.sqrt(cov.loc[t, t]), mu[t], marker="o", c="gray"); ax.annotate(t, (np.sqrt(cov.loc[t, t]), mu[t]))
ax.set_xlabel("Độ biến động năm"); ax.set_ylabel("Lợi suất kỳ vọng năm")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"Hình 6. Đường biên hiệu quả và {N_PORTFOLIOS} danh mục ngẫu nhiên"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig6_efficient_frontier.png"); plt.show()

In [ ]:
# Kiểm tra chéo bằng PyPortfolioOpt (pip install pyportfolioopt). Bỏ qua nếu chưa cài.
try:
    from pypfopt import EfficientFrontier
    ef = EfficientFrontier(mu, cov)
    ef.max_sharpe(risk_free_rate=RISK_FREE)
    print("PyPortfolioOpt:", {k: round(v, 4) for k, v in ef.clean_weights().items()})
    ef.portfolio_performance(verbose=True, risk_free_rate=RISK_FREE)
except ImportError:
    print("Chưa cài PyPortfolioOpt, bỏ qua bước kiểm tra chéo.")

## 9. Hình 7–8: trọng số và chỉ số kỳ vọng của danh mục Sharpe tối đa

In [ ]:
weights = pd.Series(w_ms, index=top).round(4).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6, 3.5))
weights.plot.bar(ax=ax, color="#55A868"); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Hình 7. Trọng số danh mục Sharpe tối đa"); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig7_weights.png"); plt.show()

summary = pd.DataFrame({
    "Bài báo": ["29,83%", "25,46%", "không ghi"],
    "Tái lập": [f"{r_ms:.2%}", f"{v_ms:.2%}", f"{s_ms:.2f}"],
}, index=["Lợi suất kỳ vọng năm", "Độ biến động năm", "Sharpe"])
display(weights.to_frame("trọng số").style.format("{:.2%}"))
print("Hình 8. Chỉ số kỳ vọng"); display(summary)
weights.to_csv(f"{OUT_DIR}/max_sharpe_weights.csv")

## 10. Hình 9: bỏ 2.000.000 VND vào danh mục

Giả định giống bài: giữ cố định trọng số mỗi ngày (tái cân bằng hằng ngày), không tính phí giao dịch.

> **Lưu ý quan trọng:** trọng số được tối ưu trên **chính** giai đoạn dùng để vẽ hình này, nên kết quả đẹp là điều tất nhiên (look-ahead bias). Phần 11 kiểm tra lại một cách công bằng.

In [ ]:
port_ret = ret5 @ weights.reindex(top).values
port_growth = INITIAL_VND * (1 + port_ret).cumprod()
ax = port_growth.plot(figsize=(10, 4), color="red", title=f"Hình 9. Giá trị {INITIAL_VND:,.0f} VND đầu tư theo danh mục Sharpe tối đa")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e6:.0f}tr"))
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig9_portfolio_growth.png"); plt.show()
print(f"Giá trị cuối kỳ: {port_growth.iloc[-1]:,.0f} VND, tức lãi {port_growth.iloc[-1] - INITIAL_VND:,.0f} VND (bài: lãi khoảng 18.146.800 VND)")

## 11. Mở rộng: kiểm tra ngoài mẫu (bài không làm)

Cách làm của bài dùng cùng một dữ liệu để **chọn mã**, **tối ưu trọng số** và **đánh giá**. Phần này tách dữ liệu theo thời gian:
- **Giai đoạn huấn luyện:** từ đầu đến hết `SPLIT`, dùng để chọn top 5 và tính trọng số.
- **Giai đoạn kiểm tra:** sau `SPLIT`, giữ nguyên trọng số và đo kết quả thực tế.

Các danh mục so sánh:
1. Sharpe tối đa (đúng phương pháp của bài)
2. Sharpe tối đa có trần 35% mỗi mã (buộc đa dạng hóa)
3. Phương sai nhỏ nhất của top 5
4. Chia đều top 5 (1/N)
5. Chia đều cả rổ VN30 (thay cho chỉ số chuẩn)

In [ ]:
SPLIT = "2021-12-31"   # đổi tùy ý; test = 2022 -> END

train_all = returns_all.loc[:SPLIT]
test_all = returns_all.loc[SPLIT:].iloc[1:]
top_tr = train_all.mean().sort_values(ascending=False).head(TOP_N).index.tolist()
tr5 = prices[top_tr].loc[:SPLIT].dropna().pct_change().dropna()
mu_tr, cov_tr = tr5.mean() * TRADING_DAYS, tr5.cov() * TRADING_DAYS
k = len(top_tr)

strategies = {
    "Sharpe tối đa (như bài)": max_sharpe(mu_tr.values, cov_tr.values),
    "Sharpe tối đa, trần 35%": max_sharpe(mu_tr.values, cov_tr.values, bounds=[(0, 0.35)] * k),
    "Phương sai nhỏ nhất": minimize(lambda w: w @ cov_tr.values @ w, np.ones(k) / k, bounds=[(0, 1)] * k,
                                    constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], method="SLSQP").x,
    "Chia đều top 5": np.ones(k) / k,
}
print("Top 5 chọn trên giai đoạn huấn luyện:", top_tr)
display(pd.DataFrame(strategies, index=top_tr).T.style.format("{:.1%}"))

def metrics(r):
    g = (1 + r).cumprod()
    years = len(r) / TRADING_DAYS
    cagr = g.iloc[-1] ** (1 / years) - 1
    vol = r.std() * np.sqrt(TRADING_DAYS)
    mdd = (g / g.cummax() - 1).min()
    return pd.Series({"Lợi suất năm (CAGR)": cagr, "Biến động năm": vol,
                      "Sharpe": (r.mean() * TRADING_DAYS - RISK_FREE) / vol, "Sụt giảm tối đa": mdd})

test5 = test_all[top_tr].fillna(0)
test_ret = {name: test5 @ w for name, w in strategies.items()}
test_ret["Chia đều VN30"] = test_all.mean(axis=1, skipna=True)
test_ret = pd.DataFrame(test_ret).dropna()

res = test_ret.apply(metrics).T
display(res.style.format({"Lợi suất năm (CAGR)": "{:.2%}", "Biến động năm": "{:.2%}", "Sharpe": "{:.2f}", "Sụt giảm tối đa": "{:.2%}"}))
res.to_csv(f"{OUT_DIR}/out_of_sample_results.csv")

ax = (INITIAL_VND * (1 + test_ret).cumprod()).plot(figsize=(10, 4.5), title=f"Giá trị {INITIAL_VND:,.0f} VND trong giai đoạn kiểm tra ({test_ret.index.min().date()} -> {test_ret.index.max().date()})")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e6:.1f}tr"))
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig10_out_of_sample.png"); plt.show()

## 12. Ghi chú khi đọc kết quả

- **Đối chiếu với bài:** con số tái lập khó khớp tuyệt đối vì bài không công bố khoảng thời gian, lãi suất phi rủi ro, cách tính lợi suất kỳ vọng hay việc giá có điều chỉnh cổ tức không. Khớp về *hình dạng* (mã nào biến động nhiều/ít, trọng số dồn vào mã nào) là đạt.
- **Phần 4–10** lặp lại đúng phương pháp của bài, nên có cùng điểm yếu: chọn mã và tối ưu trên chính dữ liệu dùng để đánh giá.
- **Phần 11** cho biết phương pháp có còn hiệu quả khi áp dụng cho tương lai hay không. Nếu "Sharpe tối đa" thua "Chia đều", đó là kết luận đáng viết trong một bài nghiên cứu tiếp theo.
- Hướng mở rộng tiếp: thêm một mô hình dự báo lợi suất thật (LSTM, XGBoost…) thay cho trung bình lịch sử, tái cân bằng định kỳ theo cửa sổ trượt, tính phí giao dịch.